In [1]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 39.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 78.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 MB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 70.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 4.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pylatexenc: filename=pylatexenc-2.10-py3-none-any.whl size=136817 sha256=6fab35f929c680a1b11f15769a2f3ba9a8c700017bb907b00cc49e278aac4c95
  Stored in directory: /root/.cache/pip/wheels/06/3e/78/fa1588c1ae991bbfd814af2bcac6cef7a178beee1939180d46
Successfully built pylatexenc


In [ ]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.

# This notebook is for a simulation of the protocol without an attacker.



In [4]:
def quantum_random_bits(n):
    qc = QuantumCircuit(n, n)
    for i in range(n):
        qc.h(i)           # Hadamard: |0> -> |+>
        qc.measure(i, i)  # Collapse to 0 or 1

    simulator = BasicSimulator()
    job       = simulator.run(transpile(qc, simulator), shots=1)
    counts    = job.result().get_counts()

    # counts has one key (one shot); it's a bit-string with MSB first, so reverse it
    bitstring = list(counts.keys())[0]
    return [int(b) for b in reversed(bitstring)]


# Show the circuit used for quantum randomness
example_rng_circuit = QuantumCircuit(4, 4)
for i in range(4):
    example_rng_circuit.h(i)
    example_rng_circuit.measure(i, i)
print('Quantum RNG circuit (4-bit example):')
print(example_rng_circuit.draw(output='text'))

In [5]:
# ── ALICE ──────────────────────────────────────────────────────────────────
N = 20

alice_bits  = quantum_random_bits(N)  # Secret bit string Alice wants to share
alice_bases = quantum_random_bits(N)  # Alice's random basis choices

print(f'Alice bits: {alice_bits}')
print(f'Alice bases: {["Z" if b==0 else "X" for b in alice_bases]}')

Alice bits  : [1, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 1, 1, 0, 0, 0, 1]
Alice bases : ['Z', 'Z', 'Z', 'Z', 'X', 'Z', 'X', 'Z', 'X', 'Z', 'X', 'Z', 'X', 'X', 'X', 'Z', 'X', 'X', 'X', 'X']


In [6]:
# ── ALICE ──────────────────────────────────────────────────────────────────
def alice_prepare_qubits(bits, bases):
    circuits = []
    for bit, basis in zip(bits, bases):
        qc = QuantumCircuit(1, 1)
        if bit == 1:
            qc.x(0)
        if basis == 1:
            qc.h(0)
        circuits.append(qc)
    return circuits


qubit_circuits = alice_prepare_qubits(alice_bits, alice_bases)

# Show the first 4 encoding circuits as examples
print('Example encoding circuits (qubits 0-3):')
for i in range(4):
    print(f'  Qubit {i}: bit={alice_bits[i]}, basis={"Z" if alice_bases[i]==0 else "X"}')
    print(qubit_circuits[i].draw(output='text'))

Example encoding circuits (qubits 0-3):
  Qubit 0: bit=1, basis=Z
     ┌───┐
  q: ┤ X ├
     └───┘
c: 1/═════
          
  Qubit 1: bit=1, basis=Z
     ┌───┐
  q: ┤ X ├
     └───┘
c: 1/═════
          
  Qubit 2: bit=0, basis=Z
     
  q: 
     
c: 1/
     
  Qubit 3: bit=0, basis=Z
     
  q: 
     
c: 1/
     
